# CRAFT Phase 3: Deployment, Quantization, and Evaluation
This notebook covers the final phase of the pipeline:
1. Merging the trained LoRA adapter weights back into the base Phi-3-Mini model.
2. Converting the merged model to GGUF format using llama.cpp.
3. Quantizing the GGUF model to Q4_K_M.
4. Running full benchmark evaluations on GSM8K, StrategyQA, and MMLU.
5. Uploading the finalized model weights directly to Hugging Face.

In [ ]:
# 1. Install dependencies
!pip install -q transformers bitsandbytes accelerate datasets peft trl loguru pyyaml scipy numpy huggingface_hub

In [ ]:
# 2. Verify files and SFT/RL checkpoints exist
import os
assert os.path.exists("checkpoints/rl/final"), "Error: checkpoints/rl/final not found. Run Phase 2 RL first!"

In [ ]:
# 3. Run quantization and merge
!python -m src.phase3_deploy.quantizer --config phi3_mini --adapter checkpoints/rl/final --output craft_output

In [ ]:
# 4. Clone and Build llama.cpp for GGUF compilation
!git clone https://github.com/ggerganov/llama.cpp.git
!cd llama.cpp && make -j

In [ ]:
# 5. Convert and quantize to Q4_K_M
!python llama.cpp/convert_hf_to_gguf.py craft_output/merged --outfile craft_output/craft_phi3_raw.gguf
!./llama.cpp/llama-quantize craft_output/craft_phi3_raw.gguf craft_output/craft_phi3_Q4_K_M.gguf Q4_K_M
print("Final model size:")
!ls -lh craft_output/craft_phi3_Q4_K_M.gguf

In [ ]:
# 6. Run benchmarks and evaluations
!python -m src.phase3_deploy.evaluator --model craft_output/craft_phi3_Q4_K_M.gguf --output craft_output/results

In [ ]:
# 7. Hugging Face Login and Upload
# Enter your HF Token in Kaggle secrets under HF_TOKEN or login below
from huggingface_hub import HfApi
import os

hf_token = os.getenv("HF_TOKEN")
if hf_token:
    api = HfApi(token=hf_token)
    print("Hugging Face API authenticated using secret token.")
    api.create_repo(repo_id="Aurvion/CRAFT-Phi3-Mini", repo_type="model", private=False, exist_ok=True)
    api.upload_file(
        path_or_fileobj="craft_output/craft_phi3_Q4_K_M.gguf",
        path_in_repo="craft_phi3_Q4_K_M.gguf",
        repo_id="Aurvion/CRAFT-Phi3-Mini"
    )
    print("GGUF model successfully uploaded to Hugging Face!")
else:
    print("HF_TOKEN environment variable not set. Please upload the GGUF model manually or login using huggingface-cli.")